# SCA-1 — Prepare pinned SigLIP2 offline asset

Downloads only `google/siglip2-base-patch16-224` at the frozen revision, verifies local-only text/image feature loading, and produces a Kaggle-uploadable ZIP whose root contains `model/` and `manifests/`.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = os.environ.get("AIC_REPO_URL", "https://github.com/Irthn1311/AIC2026_TeamPTK_SGU.git")
REPO_REF = os.environ.get("AIC_REPO_REF", "TRIAGEEG")
REPO_DIR = Path(os.environ.get("AIC_REPO_DIR", "/kaggle/working/AIC2026_TeamPTK_SGU"))
ASSET_ROOT = Path("/kaggle/working/aic2026-siglip2-base-patch16-224")
CACHE_ROOT = Path("/kaggle/working/.tmp_siglip2_download")
ZIP_PATH = Path("/kaggle/working/aic2026-siglip2-base-patch16-224.zip")
for target in (ASSET_ROOT, CACHE_ROOT):
    if target.exists():
        if Path("/kaggle/working") not in target.parents:
            raise RuntimeError(f"Refusing cleanup outside /kaggle/working: {target}")
        shutil.rmtree(target)
ZIP_PATH.unlink(missing_ok=True)
print({
    "required_inputs": {},
    "internet_required_for_repo_clone": True,
    "internet_required_for_pinned_hf_snapshot": True,
    "runtime_model_download_required": False,
    "model_id": "google/siglip2-base-patch16-224",
    "exact_revision": "0ad8c6e0ff16615356a08a1ad8c8bbc8930c434e",
    "output_zip": str(ZIP_PATH),
})


In [ ]:
if not (REPO_DIR / ".git").is_dir():
    if REPO_DIR.exists():
        raise RuntimeError(f"Incomplete repository directory: {REPO_DIR}")
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)],
        check=True,
        env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
    )
module_file = REPO_DIR / "src/triage_eg/diagnostics/sca1_siglip2_complementarity/assets.py"
if not module_file.is_file():
    raise RuntimeError("TRIAGEEG ref does not contain SCA-1 asset preparation code")
sys.path.insert(0, str(REPO_DIR / "src"))
HEAD = subprocess.run(
    ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, capture_output=True, text=True, check=True
).stdout.strip()
BRANCH = subprocess.run(
    ["git", "branch", "--show-current"], cwd=REPO_DIR, capture_output=True, text=True, check=True
).stdout.strip()
print({"branch": BRANCH, "HEAD": HEAD})


In [ ]:
from triage_eg.diagnostics.sca1_siglip2_complementarity import (
    create_asset_zip,
    local_only_load_smoke,
    prepare_offline_asset,
)

PREPARED = prepare_offline_asset(ASSET_ROOT, cache_root=CACHE_ROOT)
SMOKE = local_only_load_smoke(ASSET_ROOT)
BUNDLE = create_asset_zip(ASSET_ROOT, ZIP_PATH)
print({
    "asset_validation": PREPARED,
    "local_only_load": SMOKE,
    "download_zip": BUNDLE,
})


## Kaggle output

Download `/kaggle/working/aic2026-siglip2-base-patch16-224.zip` and upload it as a Kaggle Dataset. The SCA-1 experiment notebook performs no model download.